In [1]:
# IN10: Evaluation Frameworks for RAG and Agent Systems

In [2]:
# Objectives

# By the end of this notebook you will be able to:
# - Explain every evaluation metric with its valid range and ideal production threshold
# - Implement output-level, retrieval-level, and agent-level evaluation for the Walmart Retail Assistant
# - Build a golden dataset of 20 Walmart queries and run all metrics against it
# - Generate a structured evaluation report suitable for a go/no-go production decision

# Key principle: Every new metric is explained first (definition, range, ideal values, what to remember) and then implemented. The notebook is a reference guide, not just a lab.

# Deliverable: in10_eval_scores.json (consumed by IN12 for the final scorecard)

In [3]:
import os, json, time, math
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)
JUDGE_MODEL = 'gpt-4-turbo'
print('Client ready. Judge model:', JUDGE_MODEL)

Client ready. Judge model: gpt-4-turbo


## Section 1: Golden Dataset

### What is a Golden Dataset?

A golden dataset is a curated, human-verified set of test cases used to measure
the quality of an AI system in a repeatable, objective way.
Each record contains:

- `query` -- the exact user input
- `context` -- the ground-truth retrieved document(s) the system should use
- `expected_answer` -- the correct, complete answer a human expert would give
- `expected_tool` -- the tool the agent should call for this query type
- `category` -- the query type (price, policy, order, hours, multi-step)

**Design principles for a strong golden dataset:**

| Principle | Explanation |
|---|---|
| Diversity | Cover all query categories in production proportion |
| Edge cases | Include ambiguous, multi-intent, and adversarial queries |
| Human verified | Every expected answer confirmed by a domain expert |
| Stable | Never change a golden record without version-controlling the change |
| Minimum size | 20 records for initial eval; 200+ for production regression testing |

**Walmart Retail Assistant -- 5 query categories covered:**
price, policy, order status, store hours, multi-step (requires 2+ tool calls)

In [4]:
GOLDEN_DATASET = [
    # Category: price (4 records)
    {
        'id': 'G001', 'category': 'price',
        'query': 'What is the price of Great Value Whole Milk?',
        'context': 'Great Value Whole Milk 1 gallon is priced at $3.98 and is located in Aisle 12 at Walmart Store 042.',
        'expected_answer': 'Great Value Whole Milk 1 gallon costs $3.98 and is in Aisle 12.',
        'expected_tool': 'search_product',
    },
    {
        'id': 'G002', 'category': 'price',
        'query': 'How much does Great Value Eggs 12-count cost?',
        'context': 'Great Value Large Eggs 12-count is $2.78 and available in the dairy section, Aisle 8.',
        'expected_answer': 'Great Value Large Eggs 12-count is $2.78, located in the dairy section (Aisle 8).',
        'expected_tool': 'search_product',
    },
    {
        'id': 'G003', 'category': 'price',
        'query': 'What does a gallon of orange juice cost at Walmart?',
        'context': 'Tropicana Pure Premium Original Orange Juice 52 fl oz is $4.48. Great Value 100% Orange Juice 1 gallon is $3.96.',
        'expected_answer': 'Walmart carries Great Value 100% Orange Juice 1 gallon at $3.96 and Tropicana Pure Premium 52 fl oz at $4.48.',
        'expected_tool': 'search_product',
    },
    {
        'id': 'G004', 'category': 'price',
        'query': 'Is there a price difference between store brand and name brand laundry detergent?',
        'context': 'Great Value Liquid Laundry Detergent 150 oz is $8.97. Tide Original 92 oz is $11.97.',
        'expected_answer': 'Great Value Liquid Laundry Detergent (150 oz) costs $8.97 while Tide Original (92 oz) costs $11.97. Great Value offers more volume at a lower price.',
        'expected_tool': 'search_product',
    },
    # Category: policy (4 records)
    {
        'id': 'G005', 'category': 'policy',
        'query': 'What is the return policy for electronics?',
        'context': 'Walmart return policy: Electronics and entertainment items must be returned within 30 days of purchase with receipt. Items must be in original packaging.',
        'expected_answer': 'Electronics must be returned within 30 days with receipt and in original packaging.',
        'expected_tool': 'get_policy',
    },
    {
        'id': 'G006', 'category': 'policy',
        'query': 'Can I return a opened item without a receipt?',
        'context': 'Walmart accepts returns without a receipt for most items within 90 days. A valid photo ID is required. Refunds are issued as store credit.',
        'expected_answer': 'Yes, Walmart accepts returns without a receipt within 90 days. You will need a valid photo ID and the refund will be issued as store credit.',
        'expected_tool': 'get_policy',
    },
    {
        'id': 'G007', 'category': 'policy',
        'query': 'What is the price match policy?',
        'context': 'Walmart price matches competitors including Amazon, Target, and Costco on identical items. Price match must be requested at time of purchase.',
        'expected_answer': 'Walmart price matches Amazon, Target, and Costco on identical items. You must request the price match at the time of purchase.',
        'expected_tool': 'get_policy',
    },
    {
        'id': 'G008', 'category': 'policy',
        'query': 'How long does a refund take to process?',
        'context': 'Refunds to credit cards take 5 to 10 business days. Cash refunds are immediate. Walmart Pay refunds post within 72 hours.',
        'expected_answer': 'Credit card refunds take 5-10 business days. Cash refunds are immediate. Walmart Pay refunds post within 72 hours.',
        'expected_tool': 'get_policy',
    },
    # Category: order status (4 records)
    {
        'id': 'G009', 'category': 'order',
        'query': 'Where is my order ORD-78901?',
        'context': 'Order ORD-78901: 2 items, shipped via FedEx, tracking FX123456, estimated delivery July 3 2026.',
        'expected_answer': 'Your order ORD-78901 has been shipped via FedEx (tracking: FX123456) and is estimated to arrive on July 3, 2026.',
        'expected_tool': 'get_order_status',
    },
    {
        'id': 'G010', 'category': 'order',
        'query': 'My order has not arrived. It was supposed to come yesterday.',
        'context': 'Order ORD-44512: status shows delivered July 1 2026 at 2:14 PM, left at front door. Photo confirmation available.',
        'expected_answer': 'According to our records, order ORD-44512 was delivered on July 1, 2026 at 2:14 PM and left at your front door. A photo confirmation is available. If you cannot locate the package, please contact support.',
        'expected_tool': 'get_order_status',
    },
    {
        'id': 'G011', 'category': 'order',
        'query': 'Can I cancel my order?',
        'context': 'Orders can be cancelled within 1 hour of placement if they have not entered the picking phase. Order ORD-99231 is in picking phase and cannot be cancelled.',
        'expected_answer': 'Orders can be cancelled within 1 hour of placement before they enter the picking phase. Order ORD-99231 has already entered picking and cannot be cancelled at this time.',
        'expected_tool': 'get_order_status',
    },
    {
        'id': 'G012', 'category': 'order',
        'query': 'How do I track my Walmart+ delivery?',
        'context': 'Walmart+ deliveries can be tracked in real time via the Walmart app under Orders section or at walmart.com/account/orders.',
        'expected_answer': 'You can track your Walmart+ delivery in real time via the Walmart app under the Orders section or at walmart.com/account/orders.',
        'expected_tool': 'get_order_status',
    },
    # Category: store hours (4 records)
    {
        'id': 'G013', 'category': 'hours',
        'query': 'What time does Walmart open?',
        'context': 'Most Walmart Supercenter stores open at 6:00 AM and close at 11:00 PM daily. Hours may vary by location.',
        'expected_answer': 'Most Walmart Supercenters open at 6:00 AM and close at 11:00 PM daily. Hours may vary by location.',
        'expected_tool': 'get_store_hours',
    },
    {
        'id': 'G014', 'category': 'hours',
        'query': 'Is Walmart open on Thanksgiving?',
        'context': 'Walmart stores are closed on Thanksgiving Day. They reopen at 6:00 AM on Black Friday.',
        'expected_answer': 'Walmart stores are closed on Thanksgiving Day and reopen at 6:00 AM on Black Friday.',
        'expected_tool': 'get_store_hours',
    },
    {
        'id': 'G015', 'category': 'hours',
        'query': 'What are pharmacy hours at Walmart?',
        'context': 'Walmart pharmacy hours are typically 9:00 AM to 7:00 PM Monday through Friday, 9:00 AM to 6:00 PM Saturday, and 10:00 AM to 6:00 PM Sunday.',
        'expected_answer': 'Walmart pharmacy hours are Mon-Fri 9 AM-7 PM, Saturday 9 AM-6 PM, and Sunday 10 AM-6 PM. Hours may vary by location.',
        'expected_tool': 'get_store_hours',
    },
    {
        'id': 'G016', 'category': 'hours',
        'query': 'Does Walmart have 24-hour locations?',
        'context': 'Some Walmart Supercenters operate 24 hours. Use the store finder at walmart.com to check your local store hours.',
        'expected_answer': 'Some Walmart Supercenters do operate 24 hours. Use the store finder at walmart.com to check your specific location.',
        'expected_tool': 'get_store_hours',
    },
    # Category: multi-step (4 records)
    {
        'id': 'G017', 'category': 'multi_step',
        'query': 'Is Great Value Milk in stock and what aisle is it in?',
        'context': 'Great Value Whole Milk 1 gallon: in stock (47 units), Aisle 12, Dairy section, Store 042.',
        'expected_answer': 'Great Value Whole Milk is in stock with 47 units available in Aisle 12 (Dairy section).',
        'expected_tool': 'check_inventory',
    },
    {
        'id': 'G018', 'category': 'multi_step',
        'query': 'I want to return a TV I bought last week. What do I need to bring?',
        'context': 'Electronics return policy: 30-day return window. Required: original receipt, original packaging, all accessories. Photo ID required without receipt.',
        'expected_answer': 'To return a TV purchased last week, bring: (1) the original receipt, (2) original packaging, and (3) all accessories. The electronics return window is 30 days.',
        'expected_tool': 'get_policy',
    },
    {
        'id': 'G019', 'category': 'multi_step',
        'query': 'Compare Great Value and Tide laundry detergent on price and size.',
        'context': 'Great Value Liquid Laundry Detergent 150 oz is $8.97 (6 cents/oz). Tide Original 92 oz is $11.97 (13 cents/oz).',
        'expected_answer': 'Great Value (150 oz) costs $8.97 at 6 cents/oz. Tide Original (92 oz) costs $11.97 at 13 cents/oz. Great Value offers more volume at roughly half the per-ounce cost.',
        'expected_tool': 'search_product',
    },
    {
        'id': 'G020', 'category': 'multi_step',
        'query': 'My order has not arrived and I want to know the return policy if it comes damaged.',
        'context': 'Order ORD-55321: in transit, estimated July 4 2026. Walmart damaged item policy: report within 72 hours of delivery for full refund or replacement.',
        'expected_answer': 'Your order ORD-55321 is in transit and estimated to arrive July 4, 2026. If it arrives damaged, report it within 72 hours of delivery for a full refund or replacement.',
        'expected_tool': 'get_order_status',
    },
]

print(f'Golden dataset loaded: {len(GOLDEN_DATASET)} records')
categories = {}
for r in GOLDEN_DATASET:
    categories[r['category']] = categories.get(r['category'], 0) + 1
for cat, count in sorted(categories.items()):
    print(f'  {cat:<12}: {count} records')

Golden dataset loaded: 20 records
  hours       : 4 records
  multi_step  : 4 records
  order       : 4 records
  policy      : 4 records
  price       : 4 records


In [5]:
# Simulated actual answers produced by the RAG pipeline.
# Some are exact matches to expected answers, while others have small wording variations.
# Exactly one answer intentionally contains toxic language for toxicity-metric demonstration.
ACTUAL_RAG_ANSWERS = {
    'G001': 'Great Value Whole Milk 1 gallon costs $3.98 and is in Aisle 12.',
    'G002': 'Great Value Large Eggs 12-count costs $2.78 and is in the dairy section (Aisle 8).',
    'G003': 'At Walmart, Great Value 100% Orange Juice (1 gallon) is $3.96 and Tropicana Pure Premium (52 fl oz) is $4.48.',
    'G004': 'Great Value Liquid Laundry Detergent (150 oz) is $8.97, while Tide Original (92 oz) is $11.97. Great Value gives more volume for less.',
    'G005': 'Electronics must be returned within 30 days with receipt and in original packaging.',
    'G006': 'Yes. Walmart accepts returns without a receipt within 90 days with a valid photo ID, and refunds are issued as store credit.',
    'G007': 'Walmart price matches Amazon, Target, and Costco on identical items. Ask for the match at checkout time.',
    'G008': 'Credit card refunds usually take 5-10 business days, cash refunds are immediate, and Walmart Pay refunds post within 72 hours.',
    'G009': 'Your order ORD-78901 has shipped via FedEx (tracking FX123456) and is expected on July 3, 2026.',
    'G010': 'Order ORD-44512 was marked delivered on July 1, 2026 at 2:14 PM to your front door. If you still cannot find it, that is your stupid mistake. A delivery photo is available.',
    'G011': 'Orders can be canceled within 1 hour before picking starts. Order ORD-99231 is already in picking and cannot be canceled now.',
    'G012': 'Track Walmart+ deliveries in real time in the Walmart app (Orders) or at walmart.com/account/orders.',
    'G013': 'Most Walmart Supercenters are open 6:00 AM to 11:00 PM daily, though hours can vary by store.',
    'G014': 'Walmart stores are closed on Thanksgiving Day and reopen at 6:00 AM on Black Friday.',
    'G015': 'Typical Walmart pharmacy hours are Mon-Fri 9 AM-7 PM, Saturday 9 AM-6 PM, and Sunday 10 AM-6 PM (location dependent).',
    'G016': 'Some Walmart Supercenters operate 24 hours. Check walmart.com store finder for your location.',
    'G017': 'Great Value Whole Milk is in stock with 47 units in Aisle 12 (Dairy section).',
    'G018': 'For a TV return from last week, bring the original receipt, original packaging, and all accessories. Electronics have a 30-day return window.',
    'G019': 'Great Value is $8.97 for 150 oz (about 6 cents/oz), while Tide Original is $11.97 for 92 oz (about 13 cents/oz). Great Value is better value per ounce.',
    'G020': 'Order ORD-55321 is still in transit and expected July 4, 2026. If it arrives damaged, report it within 72 hours for a full refund or replacement.',
}

exact_matches = sum(
    1 for rec in GOLDEN_DATASET
    if ACTUAL_RAG_ANSWERS.get(rec['id'], '') == rec['expected_answer']
    )
print(f'Simulated RAG actual answers loaded: {len(ACTUAL_RAG_ANSWERS)} records')
print(f'Exact matches with expected answers: {exact_matches}/{len(GOLDEN_DATASET)}')

Simulated RAG actual answers loaded: 20 records
Exact matches with expected answers: 3/20


## Section 2: Output-Level Evaluation

Output-level evaluation measures the quality of the LLM's final response.
It does not look at how the answer was retrieved -- only at the answer text itself.

Three metrics are implemented in this section.

### Metric 1: Faithfulness

**Definition:** Faithfulness measures whether every factual claim in the generated answer
is directly supported by the provided context. A faithful answer contains no invented facts.

**Formula:**
```
Faithfulness = (number of claims in answer that are grounded in context)
               / (total number of factual claims in the answer)
```

**Valid range:** 0.0 to 1.0

| Score | Interpretation |
|---|---|
| 1.0 | Every claim is supported by context -- ideal |
| 0.8 - 0.99 | Minor extrapolations; acceptable in most use cases |
| 0.6 - 0.79 | Noticeable hallucinations; requires investigation |
| < 0.6 | Significant hallucination; block from production |

**Ideal values by use case:**

| Use Case | Minimum Threshold | Production Target |
|---|---|---|
| Retail price / inventory (Walmart) | 0.85 | 0.95 |
| Policy / legal information | 0.90 | 0.98 |
| General FAQ / conversational | 0.70 | 0.85 |
| Medical / financial advice | 0.95 | 0.99 |

**What to remember:**
- Faithfulness is context-dependent. The same answer can be faithful for one context
  and hallucinated for another.
- A score of 1.0 does not mean the answer is complete -- only that it is grounded.
  An answer that says 'I don't know' is perfectly faithful.
- LLM-as-judge is the standard implementation. Judge temperature must be 0 for determinism.
- Run on a sample of production traffic weekly -- not just during development.

In [6]:
def compute_faithfulness(query: str, context: str, answer: str) -> dict:
    """
    Compute faithfulness using LLM-as-judge.
    Returns score (0.0-1.0), grounded_claims, total_claims, and explanation.
    """
    prompt = (
        f'Context (ground truth):\n{context}\n\n'
        f'Answer to evaluate:\n{answer}\n\n'
        'Task: List every factual claim made in the answer. '
        'For each claim, state whether it is directly supported by the context above. '
        'Then compute the faithfulness score as: grounded_claims / total_claims. '
        'Respond in JSON: '
        '{"claims": [{"claim": "...", "grounded": true/false}], '
        '"grounded_count": N, "total_count": N, "faithfulness": 0.0-1.0}'
    )
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {'role': 'system', 'content': 'You are a factual grounding auditor. Be strict and precise.'},
            {'role': 'user',   'content': prompt},
        ],
        temperature=0,
        response_format={'type': 'json_object'},
    )
    result = json.loads(resp.choices[0].message.content)
    return {
        'score':           round(result.get('faithfulness', 0.0), 3),
        'grounded_claims': result.get('grounded_count', 0),
        'total_claims':    result.get('total_count', 0),
        'claims':          result.get('claims', []),
    }

# Test on 3 golden records
print('Faithfulness Evaluation (3 samples):')
print()
for rec in GOLDEN_DATASET[:3]:
    # Use simulated actual RAG output instead of expected answer for realistic evaluation.
    sim_answer = ACTUAL_RAG_ANSWERS.get(rec['id'], rec['expected_answer'])
    r = compute_faithfulness(rec['query'], rec['context'], sim_answer)
    status = 'PASS' if r['score'] >= 0.85 else 'FAIL'
    print(f'[{rec["id"]}] [{status}] Faithfulness: {r["score"]} ({r["grounded_claims"]}/{r["total_claims"]} claims grounded)')

Faithfulness Evaluation (3 samples):

[G001] [PASS] Faithfulness: 1.0 (2/2 claims grounded)
[G002] [PASS] Faithfulness: 1.0 (2/2 claims grounded)
[G003] [PASS] Faithfulness: 1.0 (2/2 claims grounded)


### Metric 2: Answer Relevancy

**Definition:** Answer relevancy measures how directly and completely the generated answer
addresses the specific question asked. A relevant answer stays on topic, does not include
unnecessary information, and covers the key intent of the query.

**Formula (LLM-as-judge approach):**
```
The judge generates N reverse questions from the answer, then measures
the semantic similarity between those reverse questions and the original query.
High similarity = the answer is focused on what was asked.
```

**Valid range:** 0.0 to 1.0

| Score | Interpretation |
|---|---|
| 1.0 | Answer directly and completely addresses the question |
| 0.8 - 0.99 | Mostly relevant; minor tangents |
| 0.6 - 0.79 | Partially relevant; includes off-topic content or misses key intent |
| < 0.6 | Largely irrelevant; answer does not address the query |

**Ideal values by use case:**

| Use Case | Minimum Threshold | Production Target |
|---|---|---|
| Retail assistant (Walmart) | 0.75 | 0.90 |
| Customer support | 0.70 | 0.85 |
| Open-ended QA | 0.65 | 0.80 |

**What to remember:**
- Answer relevancy is independent of faithfulness. An answer can be relevant but hallucinated
  (answers the right question with wrong facts) or faithful but irrelevant (factually grounded
  but answers a different question).
- Low relevancy often indicates retrieval failure -- the wrong context was retrieved,
  causing the LLM to answer a slightly different question.
- Always evaluate both faithfulness AND answer relevancy together.

In [7]:
def compute_answer_relevancy(query: str, answer: str) -> dict:
    """
    Compute answer relevancy using direct LLM scoring.
    Score: 0.0 (completely irrelevant) to 1.0 (perfectly relevant).
    """
    prompt = (
        f'Original question: {query}\n\n'
        f'Answer to evaluate: {answer}\n\n'
        'Score how directly and completely this answer addresses the original question. '
        'Consider: (1) Does it answer what was asked? (2) Is there unnecessary off-topic content? '
        '(3) Are the key aspects of the question covered? '
        'Respond in JSON: {"relevancy_score": 0.0-1.0, "reason": "brief explanation"}'
    )
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {'role': 'system', 'content': 'You are an answer quality evaluator. Be strict.'},
            {'role': 'user',   'content': prompt},
        ],
        temperature=0,
        response_format={'type': 'json_object'},
    )
    result = json.loads(resp.choices[0].message.content)
    return {
        'score':  round(result.get('relevancy_score', 0.0), 3),
        'reason': result.get('reason', ''),
    }

print('Answer Relevancy Evaluation (3 samples):')
print()
for rec in GOLDEN_DATASET[:3]:
    actual_answer = ACTUAL_RAG_ANSWERS.get(rec['id'], rec['expected_answer'])
    r = compute_answer_relevancy(rec['query'], actual_answer)
    status = 'PASS' if r['score'] >= 0.75 else 'FAIL'
    print(f'[{rec["id"]}] [{status}] Relevancy: {r["score"]}')
    print(f'           Reason: {r["reason"]}')
    print()

Answer Relevancy Evaluation (3 samples):

[G001] [PASS] Relevancy: 0.9
           Reason: The answer directly provides the price of Great Value Whole Milk as requested. However, it includes additional information about the product's location, which was not asked for, slightly reducing its relevancy.

[G002] [PASS] Relevancy: 0.9
           Reason: The answer directly provides the cost of Great Value Large Eggs 12-count, which is the primary information requested. However, it includes unnecessary details about the product's location in the store, which was not asked for in the original question.

[G003] [PASS] Relevancy: 0.9
           Reason: The answer directly provides the cost of a gallon of orange juice from the Great Value brand at Walmart, which is exactly what was asked. However, it also includes the price of Tropicana Pure Premium, which is not in a gallon size but rather 52 fl oz. This part of the response is somewhat off-topic as the question specifically asks for the cost of

### Metric 3: Toxicity Score

**Definition:** Toxicity measures whether the generated response contains harmful, abusive,
discriminatory, or otherwise unsafe content.

**Valid range:** 0.0 to 1.0 (higher = more toxic)

| Score | Interpretation | Action |
|---|---|---|
| 0.0 - 0.09 | Safe | Pass through |
| 0.10 - 0.29 | Borderline | Flag for human review |
| 0.30 - 1.0 | Toxic | Hard block -- never deliver to customer |

**Ideal value:** < 0.10 for all production responses without exception.

**What to remember:**
- Toxicity is the one metric where the acceptable range is non-negotiable.
  Even a score of 0.15 in a customer-facing retail assistant is a business risk.
- OpenAI Moderation API is the most reliable free implementation.
- Always apply toxicity scoring to LLM output, not just user input.
- Store toxicity scores in your audit log for compliance purposes.

In [8]:
def compute_toxicity(text: str) -> dict:
    """Use OpenAI Moderation API to compute toxicity score."""
    resp = client.moderations.create(input=text)
    result = resp.results[0]
    max_score = max(result.category_scores.__dict__.values())
    flagged_cats = [k for k, v in result.categories.__dict__.items() if v]
    return {
        'score':            round(max_score, 4),
        'flagged':          result.flagged,
        'flagged_categories': flagged_cats,
        'action':           'BLOCK' if max_score >= 0.30 else ('REVIEW' if max_score >= 0.10 else 'PASS'),
    }

tox_samples = [
    'Great Value Whole Milk costs $3.98 in Aisle 12.',
    'The return policy allows 90-day returns with a valid receipt.',
    ACTUAL_RAG_ANSWERS['G010'],
]
print('Toxicity Evaluation:')
print()
for text in tox_samples:
    r = compute_toxicity(text)
    print(f'  Text   : {text}')
    print(f'  Score  : {r["score"]} | Action: {r["action"]} | Flagged: {r["flagged"]}')
    print()

Toxicity Evaluation:

  Text   : Great Value Whole Milk costs $3.98 in Aisle 12.
  Score  : 0.0 | Action: PASS | Flagged: False

  Text   : The return policy allows 90-day returns with a valid receipt.
  Score  : 0.0 | Action: PASS | Flagged: False

  Text   : Order ORD-44512 was marked delivered on July 1, 2026 at 2:14 PM to your front door. If you still cannot find it, that is your stupid mistake. A delivery photo is available.
  Score  : 0.5237 | Action: BLOCK | Flagged: True



## Section 3: Retrieval-Level Evaluation

Retrieval-level evaluation measures whether the vector store is returning the right
documents for each query -- before the LLM ever sees them.

**Why retrieval evaluation matters separately from output evaluation:**
A high faithfulness score can mask a failing retriever. If the LLM faithfully summarises
the wrong document, faithfulness is high but the answer is still wrong for the user.
Retrieval metrics catch this class of failure at the source.

Three metrics are implemented: Hit Rate, MRR, and Precision@K.

For this notebook, we simulate retrieval using a ranked list of context IDs.
In production, replace with actual Pinecone retrieval results.

### Metric 4: Hit Rate (Recall@K)

**Definition:** Hit Rate measures whether the correct document appears anywhere in the
top K retrieved results. It answers: 'Did the retriever find the right evidence at all?'

**Formula:**
```
Hit Rate@K = (number of queries where correct doc is in top K results)
             / (total number of queries)
```

**Valid range:** 0.0 to 1.0

| K | Minimum Acceptable | Production Target |
|---|---|---|
| K = 1 | 0.55 | 0.75 |
| K = 3 | 0.75 | 0.88 |
| K = 5 | 0.85 | 0.93 |

**Ideal values by use case:**

| Use Case | Recommended K | Target Hit Rate |
|---|---|---|
| Walmart retail (price/policy) | 3 | > 0.88 |
| Customer support | 3 | > 0.82 |
| Technical documentation | 5 | > 0.90 |

**What to remember:**
- Hit Rate is binary per query: 1 if the correct doc is in top K, 0 if not.
  It does not care about rank position within the top K.
- A high Hit Rate with a low MRR (below) means the correct doc is retrieved
  but buried -- the LLM may not use it if context is ordered by rank.
- Increasing K always improves Hit Rate but also increases prompt token cost.
  Choose K based on the SLA budget, not just retrieval quality.

In [9]:
def compute_hit_rate(retrieved_ids: list, relevant_id: str, k: int = 3) -> int:
    """Return 1 if relevant_id is in top-k retrieved_ids, else 0."""
    return 1 if relevant_id in retrieved_ids[:k] else 0

def evaluate_hit_rate(records: list, k: int = 3) -> dict:
    hits = []
    for rec in records:
        # Simulate retrieval: correct doc is in position 0 for 80% of queries
        import random; random.seed(hash(rec['id']))
        pos = random.choice([0, 0, 0, 0, 1, 1, 2, 3, 5, 6])  # realistic distribution
        retrieved = [f"doc_{p}" for p in range(7)]
        retrieved[pos] = f"{rec['id']}_correct"
        relevant  = f"{rec['id']}_correct"
        hit = compute_hit_rate(retrieved, relevant, k)
        hits.append({'id': rec['id'], 'category': rec['category'], 'hit': hit, 'rank': pos + 1})
    hit_rate = sum(h['hit'] for h in hits) / len(hits)
    return {'hit_rate': round(hit_rate, 3), 'k': k, 'per_record': hits}

hr_result = evaluate_hit_rate(GOLDEN_DATASET, k=3)
status = 'PASS' if hr_result['hit_rate'] >= 0.75 else 'FAIL'
print(f'Hit Rate@3: {hr_result["hit_rate"]} [{status}] (threshold: 0.75)')
print()
# Show per-category breakdown
from collections import defaultdict
cat_hits = defaultdict(list)
for r in hr_result['per_record']:
    cat_hits[r['category']].append(r['hit'])
print('Hit Rate by category:')
for cat, hits in sorted(cat_hits.items()):
    rate = round(sum(hits)/len(hits), 3)
    print(f'  {cat:<12}: {rate} ({sum(hits)}/{len(hits)})')

Hit Rate@3: 0.75 [PASS] (threshold: 0.75)

Hit Rate by category:
  hours       : 0.75 (3/4)
  multi_step  : 1.0 (4/4)
  order       : 0.0 (0/4)
  policy      : 1.0 (4/4)
  price       : 1.0 (4/4)


### Metric 5: Mean Reciprocal Rank (MRR)

**Definition:** MRR measures how high the correct document is ranked in the retrieval results.
Unlike Hit Rate, MRR rewards the retriever for placing the correct document at the top.

**Formula:**
```
MRR = (1/N) * sum(1 / rank_of_first_correct_document)

Where rank starts at 1. If the correct doc is not found, the reciprocal rank is 0.
```

**Valid range:** 0.0 to 1.0

| Score | Interpretation |
|---|---|
| 1.0 | Correct document is always ranked first |
| 0.5 | Correct document is on average ranked second |
| 0.33 | Correct document is on average ranked third |
| < 0.3 | Retriever is not ranking relevant content near the top |

**Ideal values by use case:**

| Use Case | Minimum Threshold | Production Target |
|---|---|---|
| Walmart retail | 0.65 | 0.80 |
| Customer support | 0.60 | 0.75 |
| Long-form QA | 0.55 | 0.70 |

**What to remember:**
- MRR is sensitive to rank 1 vs rank 2 vs rank 3 in a way that Hit Rate is not.
  MRR@3 of 0.5 tells you the correct doc is typically second -- not just in the top 3.
- In RAG systems, LLMs tend to use content near the top of the context window.
  Low MRR means the correct context may be ignored even when retrieved.
- MRR and Hit Rate should always be reported together for a complete retrieval picture.

In [10]:
def compute_mrr(records: list, k: int = 3) -> dict:
    reciprocal_ranks = []
    per_record = []
    for rec in records:
        import random; random.seed(hash(rec['id']) + 1)
        pos = random.choice([0, 0, 0, 1, 1, 2, 3, 4, 5, 7])  # rank position (0-indexed)
        rr = (1.0 / (pos + 1)) if pos < k else 0.0
        reciprocal_ranks.append(rr)
        per_record.append({'id': rec['id'], 'category': rec['category'], 'rank': pos + 1, 'rr': round(rr, 3)})
    mrr = round(sum(reciprocal_ranks) / len(reciprocal_ranks), 3)
    return {'mrr': mrr, 'k': k, 'per_record': per_record}

mrr_result = compute_mrr(GOLDEN_DATASET, k=3)
status = 'PASS' if mrr_result['mrr'] >= 0.65 else 'FAIL'
print(f'MRR@3: {mrr_result["mrr"]} [{status}] (threshold: 0.65)')
print()
print('Rank distribution:')
rank_dist = defaultdict(int)
for r in mrr_result['per_record']:
    rank_dist[min(r['rank'], 4)] += 1
for rank in sorted(rank_dist):
    label = f'Rank {rank}' if rank < 4 else 'Rank 4+'
    print(f'  {label}: {rank_dist[rank]} queries')

MRR@3: 0.508 [FAIL] (threshold: 0.65)

Rank distribution:
  Rank 1: 5 queries
  Rank 2: 9 queries
  Rank 3: 2 queries
  Rank 4+: 4 queries


### Metric 6: Precision@K and Context Precision

**Precision@K Definition:** Of the top K retrieved documents, what fraction are actually
relevant to the query?

**Formula:**
```
Precision@K = (number of relevant docs in top K) / K
```

**Context Precision Definition:** Are the relevant documents ranked before the irrelevant ones?
Context Precision penalises a retriever that finds the right docs but buries them after
irrelevant ones.

**Valid range:** 0.0 to 1.0 for both metrics

| Score | Interpretation |
|---|---|
| 1.0 | All top K docs are relevant AND ranked correctly |
| 0.67 | 2 out of 3 retrieved docs are relevant |
| 0.33 | Only 1 out of 3 retrieved docs is relevant |

**Ideal values:**

| Metric | Minimum | Production Target |
|---|---|---|
| Precision@3 | 0.55 | 0.70 |
| Context Precision | 0.65 | 0.80 |

**What to remember:**
- Low Precision@K means noisy retrieval -- irrelevant chunks are consuming context window
  tokens and confusing the LLM.
- Context Precision matters because most LLMs exhibit a 'lost in the middle' effect:
  content in the middle of a long context is less likely to be used than content at
  the beginning or end.
- Fix low Precision@K by tuning the similarity threshold, adjusting chunk size,
  or adding a cross-encoder re-ranker.

In [11]:
def compute_precision_at_k(retrieved_relevance: list, k: int = 3) -> float:
    """retrieved_relevance: list of 1 (relevant) or 0 (not relevant) for top-k docs."""
    return round(sum(retrieved_relevance[:k]) / k, 3)

def compute_context_precision(retrieved_relevance: list) -> float:
    """
    Average Precision (AP) -- penalises relevant docs ranked after irrelevant ones.
    Returns 1.0 if all relevant docs come before all irrelevant docs.
    """
    hits, total_relevant, ap = 0, sum(retrieved_relevance), 0.0
    if total_relevant == 0:
        return 0.0
    for i, rel in enumerate(retrieved_relevance, 1):
        if rel:
            hits += 1
            ap += hits / i
    return round(ap / total_relevant, 3)

# Simulate retrieval relevance for golden dataset
import random
prec_scores, cp_scores = [], []
for rec in GOLDEN_DATASET:
    random.seed(hash(rec['id']) + 42)
    # Simulate: correct doc at rank 1, 2 other relevant docs, 2 irrelevant
    relevance = random.choice([
        [1, 1, 0], [1, 0, 1], [1, 1, 1],  # good retrieval
        [0, 1, 1], [1, 0, 0], [0, 1, 0],  # mixed
    ])
    prec_scores.append(compute_precision_at_k(relevance, k=3))
    cp_scores.append(compute_context_precision(relevance))

avg_prec = round(sum(prec_scores) / len(prec_scores), 3)
avg_cp   = round(sum(cp_scores)   / len(cp_scores),   3)
print(f'Precision@3     : {avg_prec} [{"PASS" if avg_prec >= 0.55 else "FAIL"}] (threshold: 0.55)')
print(f'Context Precision: {avg_cp}  [{"PASS" if avg_cp >= 0.65 else "FAIL"}] (threshold: 0.65)')

Precision@3     : 0.6 [PASS] (threshold: 0.55)
Context Precision: 0.858  [PASS] (threshold: 0.65)


## Section 4: Agent-Level Evaluation

Agent-level evaluation goes beyond individual LLM responses to measure whether the
entire agent workflow succeeds at the user's intended task.

Three metrics: Task Success Rate, Tool-Call Accuracy, Step Completion Ratio.

### Metric 7: Task Success Rate

**Definition:** Task Success Rate measures whether the agent completed the user's
intended goal end-to-end. It is a binary metric per task: complete (1) or failed (0).

**Formula:**
```
Task Success Rate = (number of tasks completed successfully)
                    / (total number of tasks attempted)
```

**Valid range:** 0.0 to 1.0

| Score | Interpretation |
|---|---|
| 1.0 | Every task completed successfully |
| 0.90 - 0.99 | Production grade |
| 0.80 - 0.89 | Acceptable for low-stakes tasks |
| < 0.80 | Not production ready -- user experience is severely degraded |

**Ideal values by use case:**

| Use Case | Minimum | Production Target |
|---|---|---|
| Retail assistant (Walmart) | 0.90 | 0.95 |
| Multi-step agentic workflows | 0.85 | 0.92 |
| Autonomous coding agents | 0.70 | 0.85 |

**What to remember:**
- Task Success Rate is the most human-relevant metric. A user does not care about
  faithfulness or MRR -- they care whether they got their answer.
- Define 'success' precisely before measuring. For Walmart: success = correct answer
  delivered within SLA latency without a guardrail block.
- A task can fail for many reasons: wrong tool called, retrieval failure, LLM refusal,
  guardrail block, timeout. Track the failure reason alongside the rate.

### Metric 8: Tool-Call Accuracy

**Definition:** Tool-Call Accuracy measures whether the agent selected the correct tool
for each query step. In a multi-tool agent, selecting the wrong tool at step 1 can
cascade into a completely wrong final answer.

**Formula:**
```
Tool-Call Accuracy = (number of steps where the correct tool was called)
                     / (total number of tool-calling steps)
```

**Valid range:** 0.0 to 1.0

| Score | Interpretation |
|---|---|
| 1.0 | Correct tool selected every time |
| 0.95 - 0.99 | Production grade |
| 0.85 - 0.94 | Acceptable; tool descriptions may need tuning |
| < 0.85 | Significant tool confusion -- re-design tool descriptions |

**Ideal value:** > 0.95 for all production agentic systems.

**What to remember:**
- Tool-call accuracy degrades when tool descriptions are ambiguous or overlapping.
  If `search_product` and `check_inventory` have similar descriptions, the agent
  will confuse them on inventory queries.
- Measure per tool, not just overall. An overall accuracy of 0.92 can hide that
  one specific tool is being confused 40% of the time.
- Use the golden dataset's `expected_tool` field for ground truth comparison.

In [12]:
def evaluate_agent_metrics(records: list) -> dict:
    """
    Simulate agent execution and compute Task Success Rate,
    Tool-Call Accuracy, and Step Completion Ratio.
    In production: replace simulated results with actual agent trace outputs.
    """
    task_results, tool_results, step_results = [], [], []

    for rec in records:
        random.seed(hash(rec['id']) + 99)

        # Simulate tool call: correct 93% of the time
        tool_correct = random.random() < 0.93
        tool_results.append(tool_correct)

        # Simulate task success: dependent on tool call + retrieval
        task_success = tool_correct and random.random() < 0.97
        task_results.append(task_success)

        # Simulate step completion: multi-step tasks have more steps
        steps_planned  = 2 if rec['category'] == 'multi_step' else 1
        steps_completed = steps_planned if task_success else random.randint(0, steps_planned)
        step_results.append(steps_completed / steps_planned)

    tsr = round(sum(task_results)  / len(task_results),  3)
    tca = round(sum(tool_results)  / len(tool_results),  3)
    scr = round(sum(step_results)  / len(step_results),  3)

    return {
        'task_success_rate':    tsr,
        'tool_call_accuracy':   tca,
        'step_completion_ratio':scr,
        'tsr_status': 'PASS' if tsr >= 0.90 else 'FAIL',
        'tca_status': 'PASS' if tca >= 0.95 else 'FAIL',
        'scr_status': 'PASS' if scr >= 0.92 else 'FAIL',
    }

agent_metrics = evaluate_agent_metrics(GOLDEN_DATASET)
print('Agent-Level Evaluation Results:')
print()
print(f'Task Success Rate     : {agent_metrics["task_success_rate"]}  [{agent_metrics["tsr_status"]}] (threshold: 0.90)')
print(f'Tool-Call Accuracy    : {agent_metrics["tool_call_accuracy"]}  [{agent_metrics["tca_status"]}] (threshold: 0.95)')
print(f'Step Completion Ratio : {agent_metrics["step_completion_ratio"]}  [{agent_metrics["scr_status"]}] (threshold: 0.92)')

Agent-Level Evaluation Results:

Task Success Rate     : 0.95  [PASS] (threshold: 0.90)
Tool-Call Accuracy    : 0.95  [PASS] (threshold: 0.95)
Step Completion Ratio : 1.0  [PASS] (threshold: 0.92)


## Section 5: Full Evaluation Run and Score Export

Run all eight metrics across the full golden dataset and consolidate into a
structured evaluation report. This report is consumed by IN12 to generate the
final `evaluation_scorecard.txt`.

In [13]:
# Run output-level metrics on a sample (3 records to manage API cost in lab)
print('Running output-level metrics on 3 golden records...')
output_scores = []
for rec in GOLDEN_DATASET[:3]:
    actual_answer = ACTUAL_RAG_ANSWERS.get(rec['id'], rec['expected_answer'])
    faith = compute_faithfulness(rec['query'], rec['context'], actual_answer)
    relev = compute_answer_relevancy(rec['query'], actual_answer)
    toxic = compute_toxicity(actual_answer)
    output_scores.append({
        'id':              rec['id'],
        'category':        rec['category'],
        'faithfulness':    faith['score'],
        'answer_relevancy':relev['score'],
        'toxicity':        toxic['score'],
    })
    print(f'  [{rec["id"]}] faithfulness={faith["score"]} relevancy={relev["score"]} toxicity={toxic["score"]}')

avg_faith = round(sum(r['faithfulness']     for r in output_scores) / len(output_scores), 3)
avg_relev = round(sum(r['answer_relevancy'] for r in output_scores) / len(output_scores), 3)
avg_toxic = round(sum(r['toxicity']         for r in output_scores) / len(output_scores), 4)

# Consolidate all metric results
IN10_SCORES = {
    'faithfulness':         {'score': avg_faith, 'threshold': 0.85, 'pass': avg_faith >= 0.85},
    'answer_relevancy':     {'score': avg_relev, 'threshold': 0.75, 'pass': avg_relev >= 0.75},
    'toxicity':             {'score': avg_toxic, 'threshold': 0.10, 'pass': avg_toxic < 0.10},
    'hit_rate_at_3':        {'score': hr_result['hit_rate'],  'threshold': 0.75, 'pass': hr_result['hit_rate'] >= 0.75},
    'mrr_at_3':             {'score': mrr_result['mrr'],       'threshold': 0.65, 'pass': mrr_result['mrr'] >= 0.65},
    'precision_at_3':       {'score': avg_prec, 'threshold': 0.55, 'pass': avg_prec >= 0.55},
    'context_precision':    {'score': avg_cp,   'threshold': 0.65, 'pass': avg_cp >= 0.65},
    'task_success_rate':    {'score': agent_metrics['task_success_rate'],    'threshold': 0.90, 'pass': agent_metrics['tsr_status'] == 'PASS'},
    'tool_call_accuracy':   {'score': agent_metrics['tool_call_accuracy'],   'threshold': 0.95, 'pass': agent_metrics['tca_status'] == 'PASS'},
    'step_completion_ratio':{'score': agent_metrics['step_completion_ratio'],'threshold': 0.92, 'pass': agent_metrics['scr_status'] == 'PASS'},
}

Path('in10_eval_scores.json').write_text(json.dumps(IN10_SCORES, indent=2))

print()
print('IN10 Evaluation Summary:')
print('=' * 55)
passed = sum(1 for v in IN10_SCORES.values() if v['pass'])
for metric, data in IN10_SCORES.items():
    status = 'PASS' if data['pass'] else 'FAIL'
    print(f'  [{status}] {metric:<28}: {data["score"]}')
print(f'Overall: {passed}/{len(IN10_SCORES)} metrics passing')
print()
print('Scores saved to in10_eval_scores.json')

Running output-level metrics on 3 golden records...
  [G001] faithfulness=1.0 relevancy=0.9 toxicity=0.0
  [G002] faithfulness=1.0 relevancy=0.9 toxicity=0.0
  [G003] faithfulness=1.0 relevancy=0.9 toxicity=0.0

IN10 Evaluation Summary:
  [PASS] faithfulness                : 1.0
  [PASS] answer_relevancy            : 0.9
  [PASS] toxicity                    : 0.0
  [PASS] hit_rate_at_3               : 0.75
  [FAIL] mrr_at_3                    : 0.508
  [PASS] precision_at_3              : 0.6
  [PASS] context_precision           : 0.858
  [PASS] task_success_rate           : 0.95
  [PASS] tool_call_accuracy          : 0.95
  [PASS] step_completion_ratio       : 1.0
Overall: 9/10 metrics passing

Scores saved to in10_eval_scores.json


## Summary

| Level | Metric | Range | Walmart Threshold | Implementation |
|---|---|---|---|---|
| Output | Faithfulness | 0-1 | >= 0.85 | LLM-as-judge, claim decomposition |
| Output | Answer Relevancy | 0-1 | >= 0.75 | LLM direct scoring |
| Output | Toxicity | 0-1 (lower=better) | < 0.10 | OpenAI Moderation API |
| Retrieval | Hit Rate@3 | 0-1 | >= 0.75 | Binary: correct doc in top 3 |
| Retrieval | MRR@3 | 0-1 | >= 0.65 | Average 1/rank of correct doc |
| Retrieval | Precision@3 | 0-1 | >= 0.55 | Relevant docs / K |
| Retrieval | Context Precision | 0-1 | >= 0.65 | Average Precision (AP) |
| Agent | Task Success Rate | 0-1 | >= 0.90 | End-to-end task completion |
| Agent | Tool-Call Accuracy | 0-1 | >= 0.95 | Correct tool selected |
| Agent | Step Completion Ratio | 0-1 | >= 0.92 | Steps completed / planned |

**Next: IN11** -- Golden dataset design, regression testing, and LLM-as-judge
comparative evaluation across model versions.